# Assignment 1 — Build Your Own Simple Agent

Goal: implement a functional agent that takes a user query, reasons about
what to do, and executes an action (search or calculation) using an explicit
**Thought → Action → Observation → Answer** (ReAct) loop.

This notebook is self-contained: tools, model, and the reasoning loop are
all defined here.


## 0. Setup

In [1]:
!pip install -q transformers accelerate ddgs


In [2]:
import re
import ast
import operator as op

from transformers import pipeline


## 1. The language model

We use an instruction-tuned model via `transformers.pipeline`.
`mistralai/Mistral-7B-Instruct-v0.2` (the example given in the assignment) is
**gated** on the Hugging Face Hub (you must accept its license and pass an
`HF_TOKEN`) and needs a GPU with ~16GB+ VRAM to run comfortably in fp16.

For quick local/CPU experimentation, swap `MODEL_NAME` for a small open
instruction model such as `Qwen/Qwen2.5-1.5B-Instruct` or
`HuggingFaceH4/zephyr-7b-beta` — the rest of the notebook does not change.


In [3]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
# MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # <- lighter alternative for CPU/local testing

# If MODEL_NAME is gated, log in first:
# from huggingface_hub import login
# login("hf_xxx...")

generator = pipeline(
    "text-generation",
    model=MODEL_NAME,
    max_new_tokens=300,
    do_sample=False,
    temperature=0.0,
)


def llm(prompt: str) -> str:
    """Single text-completion call. Returns only the newly generated text."""
    out = generator(prompt, max_new_tokens=300, return_full_text=False)
    return out[0]["generated_text"].strip()


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


## 2. Tools

### `search_tool` — DuckDuckGo web search
### `calculator_tool` — safe arithmetic expression evaluator (AST-based, no `eval`)


In [4]:
from ddgs import DDGS


def search_tool(query: str, max_results: int = 3) -> str:
    """Search the web with DuckDuckGo and return a few title/snippet results."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
        if not results:
            return "No results found."
        return "\n".join(f"- {r['title']}: {r['body']}" for r in results)
    except Exception as e:
        return f"Search error: {e}"


In [5]:
_OPS = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv,
    ast.Pow: op.pow, ast.USub: op.neg, ast.UAdd: op.pos,
    ast.FloorDiv: op.floordiv,
    # Note: '%' is reserved for percent notation (see _preprocess_percent below),
    # not modulo - that avoids ambiguity with expressions like '10%*69081996'.
}

PERCENT_RE = re.compile(r"(\d+(?:\.\d+)?)\s*%")


def _preprocess_percent(expression: str) -> str:
    """Turn 'N%' into '(N/100)' so the model can write percentages naturally,
    e.g. '10%*69081996' -> '(10/100)*69081996'."""
    return PERCENT_RE.sub(r"(\1/100)", expression)


def _eval_node(node):
    if isinstance(node, ast.Constant):
        if isinstance(node.value, (int, float)):
            return node.value
        raise ValueError("Only numeric constants are allowed.")
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError(f"Unsupported expression node: {ast.dump(node)}")


def calculator_tool(expression: str) -> str:
    """Safely evaluate a math expression, e.g. '23 * 47 + 1' or '10% * 200'."""
    try:
        expression = _preprocess_percent(expression)
        tree = ast.parse(expression, mode="eval")
        return str(_eval_node(tree.body))
    except Exception as e:
        return f"Error evaluating '{expression}': {e}"


TOOLS = {"search": search_tool, "calculator": calculator_tool}


In [6]:
# quick smoke tests
print(calculator_tool("23 * 47 + 1"))
print(search_tool("current population of France")[:300])


1082
- Demographics of France - Wikipedia: 1 week ago - As of 1 January 2026, in Metropolitan France 66,792,845 people lived, while 2,289,151 lived in overseas France, for a total of 69,081,996 inhabitants in the French Republic. In the 2010s and until 2017, the population of France grew by 1 million peo


## 3. The Thought → Action → Observation → Answer loop

The model is instructed to always respond in a strict ReAct format. After
each `Thought`/`Action` pair, we execute the requested tool ourselves, feed
the result back as an `Observation`, and continue until the model produces
a `Final Answer` or we hit a step limit.


In [7]:
SYSTEM_PROMPT = """You are a helpful AI agent that solves problems by reasoning step by step
using the ReAct (Reasoning + Acting) pattern.

You have access to these tools:
- search[query]: searches the web for current or factual information.
- calculator[expression]: evaluates a math expression.

Respond using EXACTLY this format, one step at a time:

Thought: <your reasoning about what to do next>
Action: <tool_name>[<tool_input>]

Wait for the Observation before continuing. When you have enough information:

Thought: I now know the final answer.
Final Answer: <the answer to the original question>

Begin!

Question: {question}
"""

ACTION_RE = re.compile(r"Action:\s*(\w+)\[(.*?)\]", re.DOTALL)


def run_agent(question: str, max_steps: int = 5, verbose: bool = True) -> str:
    transcript = SYSTEM_PROMPT.format(question=question)

    for step in range(max_steps):
        response = llm(transcript)
        # Stop at the first Observation the model might hallucinate ahead of time.
        response = response.split("Observation:")[0].strip()
        if verbose:
            print(f"--- step {step + 1} ---")
            print(response)

        transcript += "\n" + response

        if "Final Answer:" in response:
            return response.split("Final Answer:")[-1].strip()

        match = ACTION_RE.search(response)
        if not match:
            return "Agent stopped: no valid Action or Final Answer was produced."

        tool_name, tool_input = match.group(1).strip(), match.group(2).strip()
        tool_fn = TOOLS.get(tool_name)
        observation = tool_fn(tool_input) if tool_fn else f"Unknown tool '{tool_name}'."

        obs_line = f"\nObservation: {observation}\n"
        if verbose:
            print(obs_line)
        transcript += obs_line

    return "Agent stopped: reached max_steps without a Final Answer."


## 4. Example run

In [8]:
question = "What is 23 * 47, and who is the current CEO of OpenAI?"
answer = run_agent(question)
print("\n=== FINAL ANSWER ===")
print(answer)


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=

--- step 1 ---
Thought: I need to find the product of 23 and 47, and the current CEO of OpenAI.
Action: calculator[23 * 47]

Observation: 1081

--- step 2 ---
Thought: The product of 23 and 47 is 1081.
Action: search[current CEO of OpenAI]


[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Observation: - Who is the CEO of OpenAI?: Sam Altman is the CEO of OpenAI, with the main responsibility of guiding the company's mission of developing artificial general intelligence for the ...
- CEO of OpenAI – The Ubiquity: OpenAI is the creator of the AI chatbot ChatGPT, which skyrocketed the popularity of the company and the AI chatbot.
- Ex-OpenAI board member provides her first detailed account of: Sam Altman, CEO of OpenAI, attends the 54th annual meeting of the World Economic Forum, in Davos, Switzerland, January 18, 2024.

--- step 3 ---
Thought: The current CEO of OpenAI is Sam Altman.
Thought: I now know the final answer.
Final Answer: The product of 23 and 47 is 1081, and the current CEO of OpenAI is Sam Altman.

=== FINAL ANSWER ===
The product of 23 and 47 is 1081, and the current CEO of OpenAI is Sam Altman.


In [9]:
# A purely arithmetic question - should resolve via calculator_tool alone.
answer = run_agent("If a year has 365 days, how many hours is that?")
print("\n=== FINAL ANSWER ===")
print(answer)


[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- step 1 ---
Thought: There are 24 hours in a day, so to find the number of hours in a year, I need to multiply the number of days in a year by the number of hours in a day.

Action: calculator[24*365]

Observation: 8760

--- step 2 ---
Thought: I now know the final answer.
Final Answer: A year with 365 days has 8760 hours.

=== FINAL ANSWER ===
A year with 365 days has 8760 hours.


## Notes / extensions

- The regex-based parser expects the model to follow the ReAct format
  closely; less capable models may need a one-shot example added to
  `SYSTEM_PROMPT` to imitate the format reliably.
- `calculator_tool` only supports `+ - * / ** % //` on numeric literals by
  design (no `eval`, so it can't execute arbitrary code).
- This notebook intentionally avoids LangChain's built-in agent classes so
  the Thought/Action/Observation loop is fully visible and hackable; see
  Assignment 4 for a framework-based version.
